# DhakaFlow Lab 01 — SUMO Foundation (Colab-only)
All SUMO installation and real execution happen in this Colab runtime. The notebook orchestrates reusable code from `src/`; it is not the implementation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Point to the active Lab 01 after Claude merges this handoff into the thesis repository
Preferred workflow: clone/pull the thesis repo into `/content/DhakaFlow-Nexus`, then keep code in Git and outputs in Drive.


In [ ]:
from pathlib import Path
import zipfile

# MODIFIED_BY_US (merge audit 2026-08-11): the thesis repo is local-only (not on
# GitHub), so the lab is bootstrapped from a source zip stored in Drive.
# Build the zip locally with scripts/make_colab_zip.sh and upload it to
# MyDrive/DhakaFlow-Nexus-Data/Lab01_SUMO/lab01_source.zip
DRIVE_OUT = Path('/content/drive/MyDrive/DhakaFlow-Nexus-Data/Lab01_SUMO')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
LAB_ROOT = Path('/content/DhakaFlow-Nexus/paper_labs/01_sumo_foundation')
if not LAB_ROOT.exists():
    src_zip = DRIVE_OUT / 'lab01_source.zip'
    assert src_zip.exists(), (
        f'Lab source not found. Upload {src_zip.name} to {src_zip.parent} '
        '(build it locally with scripts/make_colab_zip.sh)')
    LAB_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_zip) as z:
        z.extractall(LAB_ROOT.parent)
assert LAB_ROOT.exists(), f'Lab root not found: {LAB_ROOT}'
print('LAB_ROOT=', LAB_ROOT)
print('DRIVE_OUT=', DRIVE_OUT)

## Install the complete pinned Colab runtime
This installs `eclipse-sumo==1.27.1` plus Lab 01 Python dependencies. Nothing needs to be installed on the Mac.


In [ ]:
!bash {LAB_ROOT}/scripts/install_colab.sh


In [ ]:
import sys, os
sys.path.insert(0, str(LAB_ROOT / 'src'))
from dhakaflow_lab01.colab_env import configure_sumo_env
env = configure_sumo_env()
env


In [ ]:
!python {LAB_ROOT}/scripts/verify_environment.py
!sumo --version | head -n 3
!netconvert --version | head -n 3


## Static/unit tests and deterministic network build


In [ ]:
!cd {LAB_ROOT} && pytest -q
!bash {LAB_ROOT}/scripts/build_unit_network.sh


## Smoke matrix — same demand/scenario, controller comparison


In [ ]:
!python {LAB_ROOT}/scripts/run_matrix.py --config {LAB_ROOT}/configs/smoke.yaml --output-root {DRIVE_OUT}/unit_smoke


## Full multi-seed matrix


In [ ]:
!python {LAB_ROOT}/scripts/run_matrix.py --config {LAB_ROOT}/configs/full.yaml --output-root {DRIVE_OUT}/unit_full


## Fetch pinned official sources and run Bologna/Acosta unmodified first
Upstream repositories live temporarily under `/content/dhakaflow_upstreams`; heavy evidence goes to Drive.


In [ ]:
os.environ['DHAKAFLOW_UPSTREAM_ROOT'] = '/content/dhakaflow_upstreams/lab01'
!bash {LAB_ROOT}/scripts/fetch_upstreams.sh
!bash {LAB_ROOT}/scripts/run_bologna_unmodified.sh {DRIVE_OUT}/bologna_unmodified


In [ ]:
# ===== Independent verification + evidence bundle (MODIFIED_BY_US, see resources/HANDOFF_MERGE_LOG.md) =====
import csv, json, math, shutil, statistics, subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

EV = Path('/content/evidence_lab01'); shutil.rmtree(EV, ignore_errors=True); EV.mkdir(parents=True)

def pctl(xs, p):
    xs = sorted(xs); k = (len(xs)-1)*p/100.0; f = math.floor(k); c = math.ceil(k)
    return xs[int(k)] if f == c else xs[f]*(c-k) + xs[c]*(k-f)

# --- 1) Independent metric recomputation (fresh code, not metrics.py), all controllers, seed 11 ---
rows = list(csv.DictReader(open(DRIVE_OUT/'unit_full'/'per_seed_metrics.csv')))
indep = {}
for ctrl in ('fixed_time', 'actuated', 'max_pressure'):
    rd = DRIVE_OUT/'unit_full'/ctrl/'seed_11'
    root = ET.parse(rd/'tripinfo.xml').getroot()
    dur  = [float(t.get('duration')) for t in root.findall('tripinfo')]
    wait = [float(t.get('waitingTime')) for t in root.findall('tripinfo')]
    with open(rd/'step_metrics.csv') as f:
        queues = [float(r['total_halting']) for r in csv.DictReader(f)]
    row = next(r for r in rows if r['controller'] == ctrl and r['seed'] == '11')
    assert abs(statistics.mean(dur)  - float(row['avg_travel_time']))  < 1e-6, ctrl
    assert abs(pctl(dur, 95)         - float(row['p95_travel_time']))  < 1e-6, ctrl
    assert abs(statistics.mean(wait) - float(row['avg_waiting_time'])) < 1e-6, ctrl
    assert abs(statistics.mean(queues) - float(row['mean_queue']))     < 1e-6, ctrl
    assert len(dur) == int(float(row['throughput'])), ctrl
    indep[ctrl] = {'trips': len(dur), 'avg_tt': round(statistics.mean(dur), 2),
                   'p95_tt': round(pctl(dur, 95), 2), 'avg_wait': round(statistics.mean(wait), 2),
                   'mean_queue': round(statistics.mean(queues), 2)}
print('METRIC_VERIFICATION OK — independent recomputation matches pipeline (5 metrics x 3 controllers, seed 11)')
print(json.dumps(indep, indent=1))

# --- 2) Safety verification ---
safety = {}
for ctrl in ('fixed_time', 'actuated', 'max_pressure'):
    coll = tele = runs = 0; runtime = 0.0
    for sd in sorted((DRIVE_OUT/'unit_full'/ctrl).glob('seed_*')):
        runs += 1
        logtxt = (sd/'sumo.log').read_text(errors='ignore').lower() if (sd/'sumo.log').exists() else ''
        coll += logtxt.count('collision'); tele += logtxt.count('teleport')
        runtime += json.loads((sd/'run_manifest.json').read_text())['runtime_seconds']
    safety[ctrl] = {'runs': runs, 'collision_log_mentions': coll,
                    'teleport_log_mentions': tele, 'total_runtime_s': round(runtime, 1)}
    assert coll == 0, f'collision mentions in {ctrl} logs'
# static safety already asserted by tests/unit/test_scenario_xml.py (yyrr/rryy present);
# min/max-green are code-enforced pre-decision guards (see controllers/*.py).
print('SAFETY_VERIFICATION:', json.dumps(safety, indent=1))

# --- 3) Evidence bundle (lightweight only) ---
for sub in ('unit_smoke', 'unit_full'):
    d = EV/sub; d.mkdir()
    for name in ('per_seed_metrics.csv', 'aggregate_metrics.csv', 'experiment_manifest.json'):
        shutil.copy2(DRIVE_OUT/sub/name, d/name)
bg = EV/'bologna_unmodified'; bg.mkdir()
shutil.copy2(DRIVE_OUT/'bologna_unmodified'/'manifest.txt', bg/'manifest.txt')
for name in ('stdout.log', 'stderr_and_time.log'):
    srcf = DRIVE_OUT/'bologna_unmodified'/name
    if srcf.exists():
        (bg/(name + '.tail')).write_text(''.join(srcf.read_text(errors='ignore').splitlines(keepends=True)[-60:]))
env = {'sumo_version': subprocess.run(['sumo', '--version'], capture_output=True, text=True).stdout.split(chr(10))[0],
       'pip_sumo': subprocess.run(['pip', 'show', 'eclipse-sumo'], capture_output=True, text=True).stdout.split(chr(10))[1],
       'indep_metrics': indep, 'safety': safety}
(EV/'verification_record.json').write_text(json.dumps(env, indent=2))
bundle = shutil.make_archive('/content/evidence_lab01', 'zip', EV)
shutil.copy2(bundle, DRIVE_OUT/'evidence_lab01.zip')
print('Evidence bundle:', bundle, '->', DRIVE_OUT/'evidence_lab01.zip')
from google.colab import files
files.download(bundle)


## Final evidence gate
Claude must inspect logs/metrics, independently recompute checks, copy only lightweight evidence to Git, document every modification, and complete `REPRODUCTION_REPORT.md`. Real execution is required before PASS.
